# ⚡ EV Purchase Prediction: PyTorch Embedding MLP + LightGBM Blend (Playground Series S6E9)

This notebook upgrades the simple PyTorch baseline (single split, val AUC ≈ 0.9385) into a stronger pipeline while keeping the code short and readable.

**What changed vs. the baseline**

| Baseline | This notebook | Why |
|---|---|---|
| Label-encoded categoricals fed in as plain numbers | **Learned embeddings** | Label codes (0, 1, 2...) imply a fake order between categories |
| No interaction features | **`Subsidy_Available × Range_Anxiety_Level`** as an extra categorical | Cheap way to hand the model an interaction |
| 2-layer MLP, constant LR | Deeper MLP with **BatchNorm, AdamW, OneCycle LR** | More capacity and more stable training |
| One 80/20 split | **5-fold stratified CV** | Out-of-fold AUC is a steadier estimate, and the test prediction averages 5 models |
| NN only | **LightGBM on the same folds, blended with the NN** | Trees and NNs make different mistakes, so blending usually helps AUC |

**Plan:** load & preprocess → train the NN with CV → train LightGBM with CV → blend → submission.

# 1. Setup and load data

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch
import torch.nn as nn
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")

TARGET = "Will_Buy_EV"
test_ids = test["id"]
train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

print(train.shape, test.shape)
print("positive rate:", round((train[TARGET] == "Yes").mean(), 4))

# 2. Preprocess
- Categoricals → integer codes (used as **embedding indices**, not as numbers). Encoders are fit on train+test so no category is unseen.
- Numerics → standardized (for the NN only; LightGBM uses the raw values).
- One extra categorical: the `Subsidy_Available × Range_Anxiety_Level` interaction.

In [ ]:
cat_cols = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
            "Subsidy_Available", "Range_Anxiety_Level"]
num_cols = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
            "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level"]

y = (train[TARGET] == "Yes").astype(np.float32).values
X = train.drop(columns=[TARGET]).copy()
X_test = test.copy()

# Explicit interaction feature
for df in (X, X_test):
    df["Subsidy_x_Anxiety"] = df["Subsidy_Available"].astype(str) + "_" + df["Range_Anxiety_Level"].astype(str)
cat_cols = cat_cols + ["Subsidy_x_Anxiety"]

# Label-encode categoricals (fit on train+test); keep the number of categories per column for the embeddings
cat_sizes = []
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([X[col], X_test[col]]))
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])
    cat_sizes.append(len(le.classes_))

# Scale numerics for the NN
scaler = StandardScaler()
X_num = scaler.fit_transform(X[num_cols]).astype(np.float32)
X_test_num = scaler.transform(X_test[num_cols]).astype(np.float32)
X_cat = X[cat_cols].values.astype(np.int64)
X_test_cat = X_test[cat_cols].values.astype(np.int64)

# Move everything to the device once
X_cat_t, X_num_t = torch.tensor(X_cat).to(device), torch.tensor(X_num).to(device)
X_test_cat_t, X_test_num_t = torch.tensor(X_test_cat).to(device), torch.tensor(X_test_num).to(device)
y_t = torch.tensor(y).unsqueeze(1).to(device)

print("categories per column:", dict(zip(cat_cols, cat_sizes)))
print(X_cat.shape, X_num.shape, X_test_cat.shape, X_test_num.shape)

# 3. Model
Each categorical column gets its own small embedding table. The embeddings are concatenated with the scaled numeric columns and passed through an MLP (Linear → BatchNorm → ReLU → Dropout). The output is a single logit.

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, cat_sizes, n_num, emb_dim=8, hidden=(128, 64), dropout=0.2):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(size, emb_dim) for size in cat_sizes])

        in_dim = len(cat_sizes) * emb_dim + n_num
        layers = []
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        return self.mlp(torch.cat(embs + [x_num], dim=1))


def predict(model, x_cat, x_num):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(x_cat, x_num)).cpu().numpy().ravel()


print(TabularMLP(cat_sizes, len(num_cols)))

# 4. Training function (one fold)
Mini-batch training with `BCEWithLogitsLoss`, AdamW and a OneCycle learning-rate schedule. Validation AUC is tracked every epoch and the best epoch's weights are kept.

In [ ]:
EPOCHS = 12
BATCH_SIZE = 2048
MAX_LR = 3e-3


def train_fold(train_idx, val_idx):
    model = TabularMLP(cat_sizes, len(num_cols)).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=1e-4)

    n_train = len(train_idx)
    steps_per_epoch = n_train // BATCH_SIZE  # last partial batch is dropped (BatchNorm needs > 1 row)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=MAX_LR, epochs=EPOCHS, steps_per_epoch=steps_per_epoch)

    tr = torch.tensor(train_idx, device=device)
    va = torch.tensor(val_idx, device=device)
    xc_tr, xn_tr, y_tr = X_cat_t[tr], X_num_t[tr], y_t[tr]
    xc_va, xn_va = X_cat_t[va], X_num_t[va]

    best_auc, best_state, best_probs = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=device)
        epoch_loss = 0.0

        for step in range(steps_per_epoch):
            idx = perm[step * BATCH_SIZE:(step + 1) * BATCH_SIZE]
            optimizer.zero_grad()
            loss = criterion(model(xc_tr[idx], xn_tr[idx]), y_tr[idx])
            loss.backward()
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        val_probs = predict(model, xc_va, xn_va)
        val_auc = roc_auc_score(y[val_idx], val_probs)
        if val_auc > best_auc:
            best_auc, best_probs = val_auc, val_probs
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        print(f"  epoch {epoch:2d} | train_loss {epoch_loss / steps_per_epoch:.4f} | val_auc {val_auc:.4f}")

    model.load_state_dict(best_state)
    return model, best_probs, best_auc

# 5. Neural net: 5-fold cross-validation
Each fold's model predicts its held-out rows (**out-of-fold**, OOF) and the test set. OOF AUC is the number to trust, and the test predictions are averaged across the 5 models.

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X_num, y))  # reused for LightGBM so the OOF predictions are comparable

oof_nn = np.zeros(len(y))
test_nn = np.zeros(len(X_test_num))

for fold, (train_idx, val_idx) in enumerate(folds, 1):
    print(f"Fold {fold}/{N_FOLDS}")
    model, val_probs, fold_auc = train_fold(train_idx, val_idx)
    oof_nn[val_idx] = val_probs
    test_nn += predict(model, X_test_cat_t, X_test_num_t) / N_FOLDS
    print(f"Fold {fold} best AUC: {fold_auc:.4f}\n")

print(f"NN OOF AUC: {roc_auc_score(y, oof_nn):.4f}")

# 6. LightGBM: same folds
Gradient-boosted trees are the usual strong baseline on tabular data, and they capture feature interactions on their own. The label-encoded columns are passed as native categoricals; numerics stay unscaled.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*eval_set.*")  # newer LightGBM versions warn about this argument

gbm_cols = cat_cols + num_cols
X_gbm, X_test_gbm = X[gbm_cols], X_test[gbm_cols]

oof_gbm = np.zeros(len(y))
test_gbm = np.zeros(len(X_test_gbm))

for fold, (train_idx, val_idx) in enumerate(folds, 1):
    clf = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.05, num_leaves=63,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=SEED, verbose=-1)
    clf.fit(X_gbm.iloc[train_idx], y[train_idx],
            eval_set=[(X_gbm.iloc[val_idx], y[val_idx])], eval_metric="auc",
            categorical_feature=cat_cols,
            callbacks=[lgb.early_stopping(100, verbose=False)])

    oof_gbm[val_idx] = clf.predict_proba(X_gbm.iloc[val_idx])[:, 1]
    test_gbm += clf.predict_proba(X_test_gbm)[:, 1] / N_FOLDS
    print(f"Fold {fold} AUC: {roc_auc_score(y[val_idx], oof_gbm[val_idx]):.4f} | trees: {clf.best_iteration_}")

print(f"LightGBM OOF AUC: {roc_auc_score(y, oof_gbm):.4f}")

# 7. Blend
Try a few blend weights and keep the one with the best OOF AUC.

In [ ]:
best_w, best_auc = 0.0, 0.0
for w in np.linspace(0, 1, 21):
    auc = roc_auc_score(y, w *oof_nn + (1- w) * oof_gbm)
    if auc > best_auc:
        best_w, best_auc = w, auc

print(f"NN only: {roc_auc_score(y, oof_nn):.4f}")
print(f"LightGBM only: {roc_auc_score(y, oof_gbm):.4f}")
print(f"Blend (NN weight {best_w:.2f}): {best_auc:.4f}")

# 8. Predict on test and write submission

In [ ]:
test_probs = best_w * test_nn + (1 - best_w) * test_gbm

submission = pd.DataFrame({"id": test_ids, TARGET: test_probs})
submission.to_csv("submission.csv", index=False)
submission.head()

## Wrapping Up

Pipeline: embeddings + interaction feature → BatchNorm MLP trained with 5-fold CV → LightGBM on the same folds → OOF-weighted blend → submission.

**Ideas to push it further**
- Average a few seeds per model (different `SEED` values) to cut variance
- Add XGBoost or CatBoost to the blend
- Tune LightGBM (`num_leaves`, `learning_rate`, `min_child_samples`) and the NN (`emb_dim`, `hidden`, `dropout`)
- Look for more interaction features once you know which columns matter most (LightGBM's `feature_importances_` is a quick check)

If this gave you a useful starting point, an upvote is appreciated, good luck climbing the leaderboard! 